# 23 — Constraint Ablations

**Context Demystifies Forecasting**

Notebook 13 showed that a physical/residual decomposition can make forecasting more interpretable:

```text
latent_state = physical_state + residual_state
```

This notebook asks the next question:

> When does latent structure stop helping?

We test ablations by removing components, increasing residual strength, weakening physical constraints, and introducing nonstationary drift.


## Ablation questions

| Ablation | Question |
|---|---|
| Remove physical state | What survives when structured state is absent? |
| Remove residual state | How much performance is lost? |
| Scale residual amplitude | When does residual variation dominate? |
| Weaken physical constraint | How accurate must the constrained state be? |
| Add nonstationary drift | Does decomposition still help when reality changes? |

The goal is not to make a perfect forecasting model.

The goal is to understand which part of the representation carries predictive value.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIGURES = ROOT / "figures"
RESULTS = ROOT / "results"

FIGURES.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

rng = np.random.default_rng(260616076)

print(f"ROOT: {ROOT}")
print(f"FIGURES: {FIGURES}")
print(f"RESULTS: {RESULTS}")

## Shared helpers

We reuse a transparent basis model for the physical component:

```text
physical_state ≈ trend + sin(t) + cos(t) + sin(0.5t) + cos(0.5t)
```

This gives us a controllable toy latent state without hiding the result inside a neural network.


In [ ]:
def design_matrix(x, frequency_error=0.0, include_quadratic=False):
    f1 = 1.0 + frequency_error
    f2 = 0.5 + 0.5 * frequency_error

    columns = [
        np.ones_like(x),
        x,
        np.sin(f1 * x),
        np.cos(f1 * x),
        np.sin(f2 * x),
        np.cos(f2 * x),
    ]

    if include_quadratic:
        columns.append(x ** 2)

    return np.column_stack(columns)

def make_system(
    n=900,
    residual_scale=1.0,
    nonstationary=False,
    seed=260616076,
):
    local_rng = np.random.default_rng(seed)
    t = np.linspace(0, 18 * np.pi, n)

    trend = 0.010 * t
    if nonstationary:
        trend = trend + 0.00035 * t**2

    physical = (
        1.0 * np.sin(t)
        + 0.35 * np.sin(0.5 * t + 0.7)
        + trend
    )

    residual = residual_scale * (
        0.08 * local_rng.normal(size=n)
        + 0.04 * np.sin(3.7 * t + 0.4)
    )

    observed = physical + residual

    return pd.DataFrame({
        "t": t,
        "physical": physical,
        "residual": residual,
        "observed": observed,
    })

def fit_physical(history_t, history_y, frequency_error=0.0, include_quadratic=False):
    X = design_matrix(
        history_t,
        frequency_error=frequency_error,
        include_quadratic=include_quadratic,
    )
    coef, *_ = np.linalg.lstsq(X, history_y, rcond=None)

    def predict(x):
        return design_matrix(
            x,
            frequency_error=frequency_error,
            include_quadratic=include_quadratic,
        ) @ coef

    return predict, coef

def residual_mean_forecast(history_residual, horizon, window=40):
    return np.full(horizon, np.mean(history_residual[-window:]))

def residual_ar_forecast(history_residual, horizon, window=80, damping=0.92):
    r = np.asarray(history_residual[-window:])
    x = r[:-1]
    y = r[1:]

    if np.dot(x, x) < 1e-12:
        phi = 0.0
    else:
        phi = float(np.dot(x, y) / np.dot(x, x))

    phi = float(np.clip(phi, -0.95, 0.95)) * damping

    forecast = np.zeros(horizon)
    forecast[0] = phi * history_residual[-1]

    for i in range(1, horizon):
        forecast[i] = phi * forecast[i - 1]

    return forecast, phi

def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))

def cumulative_rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    sq_error = (y_true - y_pred) ** 2
    return np.sqrt(np.cumsum(sq_error) / np.arange(1, len(sq_error) + 1))

def evaluate_system(
    df,
    train_end=620,
    frequency_error=0.0,
    include_quadratic=False,
):
    history = df.iloc[:train_end].copy()
    future = df.iloc[train_end:].copy()

    history_t = history["t"].to_numpy()
    future_t = future["t"].to_numpy()
    history_y = history["observed"].to_numpy()
    truth_y = future["observed"].to_numpy()

    predict_physical, coef = fit_physical(
        history_t,
        history_y,
        frequency_error=frequency_error,
        include_quadratic=include_quadratic,
    )

    hist_physical = predict_physical(history_t)
    fut_physical = predict_physical(future_t)

    hist_residual = history_y - hist_physical
    residual_mean = residual_mean_forecast(hist_residual, len(future_t))
    residual_ar, phi = residual_ar_forecast(hist_residual, len(future_t))

    forecasts = pd.DataFrame({
        "t": future_t,
        "truth": truth_y,
        "physical_only": fut_physical,
        "residual_only": residual_mean,
        "physical_plus_residual_mean": fut_physical + residual_mean,
        "physical_plus_residual_ar": fut_physical + residual_ar,
    })

    method_cols = [
        "physical_only",
        "residual_only",
        "physical_plus_residual_mean",
        "physical_plus_residual_ar",
    ]

    metrics = pd.DataFrame([
        {
            "method": col,
            "MAE": mae(forecasts["truth"], forecasts[col]),
            "RMSE": rmse(forecasts["truth"], forecasts[col]),
        }
        for col in method_cols
    ])

    horizon = pd.DataFrame({
        "horizon": np.arange(1, len(forecasts) + 1)
    })

    for col in method_cols:
        horizon[col] = cumulative_rmse(forecasts["truth"], forecasts[col])

    return {
        "forecasts": forecasts,
        "metrics": metrics,
        "horizon": horizon,
        "residual_phi": phi,
        "coef": coef,
    }

## Baseline ablation

We begin with the same kind of toy system used in Notebook 13.

This gives us the reference result for:

```text
physical only
residual only
physical + residual mean
physical + residual AR
```


In [ ]:
base_df = make_system(residual_scale=1.0, nonstationary=False)
base_result = evaluate_system(base_df)

base_metrics = base_result["metrics"]
base_forecasts = base_result["forecasts"]
base_horizon = base_result["horizon"]

base_metrics.to_csv(RESULTS / "23_base_ablation_metrics.csv", index=False)
base_forecasts.to_csv(RESULTS / "23_base_ablation_forecasts.csv", index=False)

base_metrics

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))

ax.plot(base_forecasts["t"], base_forecasts["truth"], label="truth")
ax.plot(base_forecasts["t"], base_forecasts["physical_only"], label="physical only")
ax.plot(base_forecasts["t"], base_forecasts["residual_only"], label="residual only")
ax.plot(base_forecasts["t"], base_forecasts["physical_plus_residual_ar"], label="physical + residual AR")

ax.set_title("Ablation: remove physical state vs remove residual state")
ax.set_xlabel("time")
ax.set_ylabel("value")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "23_ablation_rollouts.png", dpi=160)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

x = np.arange(len(base_metrics))
ax.bar(x, base_metrics["RMSE"])
ax.set_xticks(x)
ax.set_xticklabels(base_metrics["method"], rotation=25, ha="right")

ax.set_title("Ablation RMSE")
ax.set_ylabel("RMSE")

fig.tight_layout()
fig.savefig(FIGURES / "23_ablation_rmse.png", dpi=160)
plt.show()

## Residual scale sweep

Now we increase the residual amplitude.

Question:

> When does residual variation become large enough that physical structure alone is no longer sufficient?


In [ ]:
residual_scales = [0.1, 0.25, 0.5, 1.0, 2.0, 4.0]

rows = []

for scale in residual_scales:
    df = make_system(residual_scale=scale, nonstationary=False, seed=260616076)
    result = evaluate_system(df)
    metrics = result["metrics"].copy()
    metrics["residual_scale"] = scale
    rows.append(metrics)

scale_metrics = pd.concat(rows, ignore_index=True)
scale_metrics.to_csv(RESULTS / "23_residual_scale_sweep.csv", index=False)

scale_metrics.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

for method in scale_metrics["method"].unique():
    subset = scale_metrics[scale_metrics["method"] == method]
    ax.plot(
        subset["residual_scale"],
        subset["RMSE"],
        marker="o",
        label=method,
    )

ax.set_title("Residual scale sweep")
ax.set_xlabel("residual scale")
ax.set_ylabel("RMSE")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "23_residual_scale_sweep.png", dpi=160)
plt.show()

## Weaken physical constraints

A latent constraint only helps if the constrained state is close enough to the system.

Here we intentionally use the wrong frequencies in the physical basis.

Question:

> How accurate must the physical constraint be before it helps?


In [ ]:
frequency_errors = [0.0, 0.01, 0.03, 0.05, 0.10, 0.20]

rows = []

for error in frequency_errors:
    df = make_system(residual_scale=1.0, nonstationary=False, seed=260616076)
    result = evaluate_system(df, frequency_error=error)
    metrics = result["metrics"].copy()
    metrics["frequency_error"] = error
    rows.append(metrics)

constraint_metrics = pd.concat(rows, ignore_index=True)
constraint_metrics.to_csv(RESULTS / "23_constraint_strength.csv", index=False)

constraint_metrics.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

for method in constraint_metrics["method"].unique():
    subset = constraint_metrics[constraint_metrics["method"] == method]
    ax.plot(
        subset["frequency_error"],
        subset["RMSE"],
        marker="o",
        label=method,
    )

ax.set_title("Constraint accuracy sweep")
ax.set_xlabel("frequency error in physical basis")
ax.set_ylabel("RMSE")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "23_constraint_strength.png", dpi=160)
plt.show()

## Nonstationary drift

Now the system changes its trend over time:

```text
trend = linear + quadratic drift
```

We compare two physical estimators:

1. a linear physical basis
2. a quadratic physical basis

Question:

> Does decomposition still help when reality changes?


In [ ]:
nonstationary_df = make_system(
    residual_scale=1.0,
    nonstationary=True,
    seed=260616076,
)

linear_result = evaluate_system(
    nonstationary_df,
    include_quadratic=False,
)

quadratic_result = evaluate_system(
    nonstationary_df,
    include_quadratic=True,
)

linear_metrics = linear_result["metrics"].copy()
linear_metrics["physical_basis"] = "linear_trend"

quadratic_metrics = quadratic_result["metrics"].copy()
quadratic_metrics["physical_basis"] = "quadratic_trend"

nonstationary_metrics = pd.concat(
    [linear_metrics, quadratic_metrics],
    ignore_index=True,
)

nonstationary_metrics.to_csv(RESULTS / "23_nonstationary_drift_metrics.csv", index=False)

nonstationary_metrics

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))

linear_forecasts = linear_result["forecasts"]
quadratic_forecasts = quadratic_result["forecasts"]

ax.plot(linear_forecasts["t"], linear_forecasts["truth"], label="truth")
ax.plot(linear_forecasts["t"], linear_forecasts["physical_plus_residual_ar"], label="linear physical basis")
ax.plot(quadratic_forecasts["t"], quadratic_forecasts["physical_plus_residual_ar"], label="quadratic physical basis")

ax.set_title("Nonstationary drift ablation")
ax.set_xlabel("time")
ax.set_ylabel("value")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "23_nonstationary_drift.png", dpi=160)
plt.show()

## Horizon stability comparison

Finally, compare cumulative error for the baseline decomposition and the nonstationary drift variants.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

ax.plot(
    base_horizon["horizon"],
    base_horizon["physical_plus_residual_ar"],
    label="stationary: physical + residual AR",
)

ax.plot(
    linear_result["horizon"]["horizon"],
    linear_result["horizon"]["physical_plus_residual_ar"],
    label="nonstationary: linear basis",
)

ax.plot(
    quadratic_result["horizon"]["horizon"],
    quadratic_result["horizon"]["physical_plus_residual_ar"],
    label="nonstationary: quadratic basis",
)

ax.set_title("Horizon stability under drift")
ax.set_xlabel("forecast horizon")
ax.set_ylabel("cumulative RMSE")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "23_horizon_stability.png", dpi=160)
plt.show()

## Summary table

We collect the main conclusions into one compact table.


In [ ]:
summary_rows = []

# Best methods from baseline.
best_base = base_metrics.sort_values("RMSE").iloc[0]
summary_rows.append({
    "experiment": "base_ablation",
    "finding": f"best method: {best_base['method']}",
    "RMSE": best_base["RMSE"],
})

# Residual scale threshold, based on where physical-only RMSE exceeds 0.25.
physical_scale = scale_metrics[scale_metrics["method"] == "physical_only"].copy()
above = physical_scale[physical_scale["RMSE"] > 0.25]

summary_rows.append({
    "experiment": "residual_scale",
    "finding": "physical-only RMSE first exceeds 0.25 at residual_scale="
               + (str(float(above.iloc[0]["residual_scale"])) if len(above) else "not reached"),
    "RMSE": float(above.iloc[0]["RMSE"]) if len(above) else float(physical_scale["RMSE"].max()),
})

# Constraint accuracy threshold.
latent_constraint = constraint_metrics[constraint_metrics["method"] == "physical_plus_residual_ar"].copy()
above = latent_constraint[latent_constraint["RMSE"] > 0.5]

summary_rows.append({
    "experiment": "constraint_accuracy",
    "finding": "latent-constrained RMSE first exceeds 0.5 at frequency_error="
               + (str(float(above.iloc[0]["frequency_error"])) if len(above) else "not reached"),
    "RMSE": float(above.iloc[0]["RMSE"]) if len(above) else float(latent_constraint["RMSE"].max()),
})

# Nonstationary comparison.
ns_best = nonstationary_metrics.sort_values("RMSE").iloc[0]
summary_rows.append({
    "experiment": "nonstationary_drift",
    "finding": f"best method/basis: {ns_best['method']} with {ns_best['physical_basis']}",
    "RMSE": ns_best["RMSE"],
})

summary = pd.DataFrame(summary_rows)
summary.to_csv(RESULTS / "23_summary.csv", index=False)

summary

## Interpretation

The ablations clarify the engineering lesson.

```text
Physical structure carries the long-horizon forecast.
Residual state improves fit only when residual variation is predictable.
Weak constraints help less as the physical basis becomes inaccurate.
Nonstationary systems require the constraint itself to be refined.
```

This prepares the repo for a real dataset.

Notebook 17 should now ask:

> Can the same decomposition survive outside a toy system?


## Next

Notebook 17 will move toward climate forecasting.

The goal is not to claim that a simple basis model solves climate prediction.

The goal is to test whether the same distinction still helps:

```text
output correction
vs
latent-state constraint
```

on real time-series data.
